# Bathing Water Quality in Europe (1990–2024)

This notebook builds the dataset and all charts used in the report *"Bathing Water Quality in
Europe (1990–2024)"*. It combines three public sources (EEA bathing water assessments, World
Bank GDP data, and Eurostat coastal tourism data), cleans and merges them, and then produces
the visualizations discussed in the report.

## 1. Imports

Install and import all the packages used throughout the notebook: data handling (`pandas`,
`numpy`), plotting (`matplotlib`, `seaborn`, `plotly`), geospatial data (`geopandas`), and
country-code utilities (`pycountry`).

In [ ]:
# ============================================================
# CELL 1: Imports
# ============================================================
!pip install pycountry geopandas plotly pyvis -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import geopandas as gpd
import networkx as nx
import pycountry
import requests
import io

## 2. Load Data

Download the main EEA bathing water dataset (1990–2024) together with the World Bank GDP
tables (current USD and PPP) directly from the project's GitHub repository.

In [ ]:
# ============================================================
# CELL 2: Load Data
# ============================================================
BASE_RAW = "https://raw.githubusercontent.com/S4mYo/data-visualisation-project/main/"

# Beach data (via requests because of the file size and the Excel format)
response = requests.get(BASE_RAW + "bw_assessment_eea_datahub_1990_2024.xlsx")
table = pd.read_excel(io.BytesIO(response.content), sheet_name="BWD_data_1990_2024")

# World Bank GDP
gdp_usd_raw = pd.read_csv(BASE_RAW + "API_NY.GDP.PCAP.CD_DS2_en_csv_v2_207559.csv", skiprows=4)
gdp_pps_raw = pd.read_csv(BASE_RAW + "API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_175516.csv", skiprows=4)

print(f"Beaches: {table.shape}, GDP USD: {gdp_usd_raw.shape}, GDP PPP: {gdp_pps_raw.shape}")

## 3. Process GDP Data and Merge Datasets

Convert the World Bank GDP tables from wide to long format, map ISO3 country codes to the
two-letter Eurostat codes used by the EEA dataset, and merge everything into a single
`final_df` table.

In [ ]:
# ============================================================
# CELL 3: Process GDP Data and Merge Datasets
# ============================================================

# Manual exceptions: ISO3 codes that differ from the Eurostat standard
COUNTRY_MAP = {'GRC': 'EL', 'GBR': 'UK', 'MNE': 'ME', 'ALB': 'AL'}

def iso3_to_eurostat(iso3):
    """ISO3 -> two-letter Eurostat country code."""
    if iso3 in COUNTRY_MAP:
        return COUNTRY_MAP[iso3]
    country = pycountry.countries.get(alpha_3=iso3)
    return country.alpha_2 if country else None

def process_wb_gdp(df_raw, value_col):
    """World Bank CSV -> long format: countryCode | season | value."""
    year_cols = [c for c in df_raw.columns if str(c).isdigit() and len(str(c)) == 4]
    df = df_raw[['Country Code'] + year_cols].copy()
    df['countryCode'] = df['Country Code'].apply(iso3_to_eurostat)
    df = df.dropna(subset=['countryCode'])
    df = df.melt(id_vars=['countryCode'], value_vars=year_cols,
                 var_name='season', value_name=value_col)
    df['season'] = pd.to_numeric(df['season'])
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    return df[['countryCode', 'season', value_col]]

# Merge USD and PPP into one table, then join it to the beach data
gdp = pd.merge(
    process_wb_gdp(gdp_usd_raw, 'gdp_usd'),
    process_wb_gdp(gdp_pps_raw, 'gdp_pps'),
    on=['countryCode', 'season'], how='outer'
)
final_df = pd.merge(table, gdp, on=['countryCode', 'season'], how='left')

print(f"Final dataset: {final_df.shape}")
print(f"Countries: {sorted(final_df['countryCode'].unique())}")
print(f"Missing GDP - usd: {final_df['gdp_usd'].isna().sum()}, pps: {final_df['gdp_pps'].isna().sum()}")

## 4. Clean the Quality Column

Map the raw `quality` text values to a numeric score (1 = Excellent to 4 = Poor) and to a clean
English label, so they can be used consistently in the aggregations and charts that follow.

In [ ]:
# ============================================================
# CELL 4: Clean the Quality Column
# ============================================================
QUALITY_MAP = {
    '1 - Excellent':          1,
    '2 - Good':               2,
    '3 - Good or Sufficient': 2.5,
    '3 - Sufficient':         3,
    '4 - Poor':               4,
    '0 - Not classified':     np.nan
}
LABEL_MAP = {
    '1 - Excellent':          'Excellent',
    '2 - Good':               'Good',
    '3 - Good or Sufficient': 'Good or Sufficient',
    '3 - Sufficient':         'Sufficient',
    '4 - Poor':               'Poor'
}

final_df['quality_score'] = final_df['quality'].map(QUALITY_MAP)
final_df['quality_label'] = final_df['quality'].map(LABEL_MAP)

print(final_df['quality_label'].value_counts())
print(f"Unclassified (NaN): {final_df['quality_score'].isna().sum()}")

## 5. Geographically Constrained Waters

A quick look at sites flagged as geographically constrained (hard to access), and how their
quality ratings are distributed, to check whether they need special handling later on.

In [ ]:
# ============================================================
# CELL 5: Constrained-Water Quality
# ============================================================

# Geographically constrained (hard-to-access) waters
constrained = final_df[final_df['geographicalConstraint'] == 'TRUE']

print(constrained.groupby('countryCode').size())
print(f"\nQuality for constrained waters:\n{constrained['quality'].value_counts()}")

## 6. Bar Chart: Countries by % Excellent (2024)

For each country, compute the percentage of bathing sites in each quality category in 2024 and
plot a horizontal stacked bar chart, ranked by the share of Excellent-rated beaches
(this is Figure 1 in the report).

In [ ]:
# ============================================================
# CELL 6: Bar Chart - Countries by % Excellent (2024)
# ============================================================
COUNTRY_NAMES = {
    'AL': 'Albania', 'AT': 'Austria', 'BE': 'Belgium', 'BG': 'Bulgaria',
    'CH': 'Switzerland', 'CY': 'Cyprus', 'CZ': 'Czechia', 'DE': 'Germany',
    'DK': 'Denmark', 'EE': 'Estonia', 'EL': 'Greece', 'ES': 'Spain',
    'FI': 'Finland', 'FR': 'France', 'HR': 'Croatia', 'HU': 'Hungary',
    'IE': 'Ireland', 'IT': 'Italy', 'LT': 'Lithuania', 'LU': 'Luxembourg',
    'LV': 'Latvia', 'ME': 'Montenegro', 'MT': 'Malta', 'NL': 'Netherlands',
    'PL': 'Poland', 'PT': 'Portugal', 'RO': 'Romania', 'SE': 'Sweden',
    'SI': 'Slovenia', 'SK': 'Slovakia', 'UK': 'United Kingdom'
}

df = final_df.dropna(subset=['quality_score']).copy()
df_2024 = df[df['season'] == 2024]

categories = ['Excellent', 'Good', 'Sufficient', 'Poor']
colors     = ['#2ecc71',   '#3498db', '#f39c12',    '#e74c3c']

country_pct = (
    df_2024.groupby(['countryCode', 'quality_label'])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=categories, fill_value=0)
)

country_pct = country_pct.div(country_pct.sum(axis=1), axis=0) * 100
country_pct = country_pct.sort_values('Excellent', ascending=True)

country_pct.index = [COUNTRY_NAMES.get(c, c) for c in country_pct.index]

ax = country_pct.plot(kind='barh', stacked=True, figsize=(12, 8),
                      color=colors, width=0.8)

plt.xlabel("% beaches")
plt.title("Quality of bathing water based on country (2024)")
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig("quality_by_country.png", dpi=300, bbox_inches='tight')
plt.show()

## 7. Water Quality by Type Over Time

For each year and water type (Coastal, Inland, Transitional), compute the percentage share of
each quality category and draw three side-by-side stacked area charts sharing the same y-axis
so they are easy to compare (Figure 10 in the report).

In [ ]:
# ============================================================
# CELL 7: Water Quality by Type Over Time (stacked area)
# ============================================================

# Lookup table: internal water-type names -> readable categories.
# Lakes and rivers are merged into a single 'Inland' category.
TYPE_MAP = {
    'coastalBathingWater':      'Coastal',
    'lakeBathingWater':         'Inland',
    'riverBathingWater':        'Inland',
    'transitionalBathingWater': 'Transitional'
}
ORDER  = ['Excellent', 'Good', 'Good or Sufficient', 'Sufficient', 'Poor']
COLORS = ['#2ecc71', '#3498db', '#1a6fa8', '#f39c12', '#e74c3c']

# Translate the water types and drop rows without a quality rating
df7 = final_df.dropna(subset=['quality_label']).copy()
df7['bathingWaterType'] = df7['bathingWaterType'].map(TYPE_MAP)

# -------------------------------------------------------
# PERCENTAGE CALCULATION
# For each year and water type, compute what percentage
# each quality category represents.
# groupby + size()        -> count of measurements per combination
# transform(x / x.sum()) -> divide by the sum for that year/type
#                           (transform keeps the index, so we
#                           don't lose columns on reset_index below)
# * 100                   -> convert to percent
# -------------------------------------------------------
pct = (
    df7.groupby(['season', 'bathingWaterType', 'quality_label'])
       .size()
       .groupby(level=[0, 1]).transform(lambda x: x / x.sum() * 100)
       .reset_index(name='percent')
)

# -------------------------------------------------------
# CHARTS
# sharey=True -> all three charts share the Y axis (0-100%)
#               making them easier to compare visually
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

for ax, wtype in zip(axes, ['Coastal', 'Inland', 'Transitional']):
    subset = pct[pct['bathingWaterType'] == wtype]

    # Hide the panel if there is no data for this type
    if subset.empty:
        ax.set_visible(False)
        continue

    # pivot -> rows = years, columns = quality categories
    # reindex -> ensures correct column order and fills 0
    #           if a category is missing for this type
    (subset.pivot(index='season', columns='quality_label', values='percent')
           .reindex(columns=ORDER, fill_value=0)
           .plot(kind='area', ax=ax, color=COLORS, alpha=0.85, legend=False))

    ax.set_title(wtype, fontsize=14)
    ax.set_ylabel("Percent (%)" if ax == axes[0] else "")
    ax.set_ylim(0, 100)

    # Show only years divisible by 5 so labels don't overlap
    years = sorted(subset['season'].unique())
    ticks = [y for y in years if y % 5 == 0]
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticks, rotation=45)
    ax.set_xlim(min(years), max(years))

# -------------------------------------------------------
# LEGEND
# Add it only to the last chart and shift it outside
# the right edge using bbox_to_anchor so it doesn't overlap the data.
# Rectangle() draws colored squares instead of lines
# (area charts would otherwise show lines, not filled patches, in the legend).
# -------------------------------------------------------
handles = [plt.Rectangle((0,0), 1, 1, color=c) for c in COLORS]
axes[2].legend(handles, ORDER, title="Quality",
               loc='center left', bbox_to_anchor=(1.02, 0.5))

fig.suptitle("Bathing Water Quality by Type (1990–2024)", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("quality_by_water_type.png", dpi=300, bbox_inches='tight')
plt.show()

## 8. Relationship Between GDP and Water Quality

Average GDP (PPP) per capita against the average water quality score per country in 2024,
shown as a scatter plot with a linear regression trend line and confidence band
(Figure 11 in the report).

In [ ]:
# ============================================================
# CELL 8: Relationship Between GDP and Water Quality
# ============================================================
plt.figure(figsize=(10, 6))
df_plot = final_df[final_df['season'] == 2024].groupby('countryCode').agg({
    'gdp_pps': 'mean',
    'quality_score': 'mean'
}).dropna()

sns.regplot(data=df_plot, x='gdp_pps', y='quality_score',
            scatter_kws={'s': 100, 'alpha': 0.6}, line_kws={'color': 'red'})
plt.title("Relationship Between GDP and Average Water Quality Score (2024)")
plt.xlabel("GDP per Capita")
plt.ylabel("Average Score")
plt.savefig("gdp_vs_quality.png", dpi=300, bbox_inches='tight')
plt.show()

## 9. Side-by-Side Map Comparison for Two Years

Define a reusable function that plots bathing-site locations on a map for two chosen years side
by side, colored by quality category, then use it to compare 2000 and 2024 across Europe
(Figure 5 in the report).

In [ ]:
# ============================================================
# CELL 9: Map Comparison for Two Years (side by side)
# ============================================================

def plot_water_quality_comparison(df, year1, year2):
    ORDER  = ['Excellent', 'Good', 'Good or Sufficient', 'Sufficient', 'Poor']
    COLORS = ['#2ecc71', '#3498db', '#1a6fa8', '#f39c12', '#e74c3c']

    # -------------------------------------------------------
    # MAIN MAP
    # px.scatter_geo plots points on a map based on lat/lon.
    # facet_col='season' automatically splits the chart into 2 panels
    # (one per year) side by side.
    # color_discrete_map assigns a specific color to each category.
    # category_orders keeps the color/year order consistent.
    # showlegend=False hides the default legend with small dots
    # - we replace it with a custom one with bigger dots below.
    # -------------------------------------------------------
    fig = px.scatter_geo(
        df[df['season'].isin([year1, year2])],
        lat='lat', lon='lon',
        color='quality_label',
        facet_col='season',
        color_discrete_map=dict(zip(ORDER, COLORS)),
        category_orders={'quality_label': ORDER, 'season': [year1, year2]},
        hover_data=['bathingWaterType'],
        width=1500, height=700
    )

    # Focus the map on Europe and shrink the markers on the map
    fig.update_geos(lonaxis_range=[-18, 40], lataxis_range=[20, 70], showcountries=True)
    fig.update_traces(marker_size=3, showlegend=False)

    # -------------------------------------------------------
    # CUSTOM LEGEND WITH LARGER DOTS
    # Plotly doesn't let you change marker size only in the legend.
    # Workaround: add invisible "dummy" traces (lat/lon=None
    # means no point is drawn on the map) just for the legend.
    # marker size=12 gives large, clearly visible dots in the legend.
    # -------------------------------------------------------
    for label, color in zip(ORDER, COLORS):
        fig.add_trace(go.Scattergeo(
            lat=[None], lon=[None],
            mode='markers',
            marker=dict(size=12, color=color),
            name=label,
            showlegend=True
        ))

    # -------------------------------------------------------
    # PANEL TITLE FORMATTING
    # px.scatter_geo automatically generates titles in the format
    # "season=2000" - for_each_annotation reformats them
    # into the more readable "Year 2000".
    # split('=')[-1] takes the part after the '=' sign, i.e. the year itself.
    # -------------------------------------------------------
    fig.for_each_annotation(lambda a: a.update(text=f"Year {a.text.split('=')[-1]}"))

    fig.update_layout(
        title=dict(text=f"Water Quality Comparison ({year1} vs {year2})",
                   y=0.95, x=0.5, xanchor='center'),
        margin=dict(l=10, r=10, t=80, b=10),
        legend=dict(
            title_text='', x=0.01, y=0.98,
            xanchor='left', yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='rgba(0,0,0,0.1)',
            borderwidth=1, font=dict(size=14)
        )
    )
    fig.show()


plot_water_quality_comparison(final_df, 2000, 2024)

## 10. Choropleth Map: % Excellent Beaches (2024)

Build a country-level choropleth map showing the percentage of beaches rated Excellent in 2024,
using country boundary polygons and a red-yellow-green color scale (Figure 2 in the report).

In [ ]:
# ============================================================
# CELL 10: Choropleth Map - % Excellent Beaches (2024)
# ============================================================

countries = gpd.read_file("https://fmfi-compbio.github.io/viz/data/country_boundaries.geojson")
countries = countries.set_index('ISO3')

ALPHA2_TO_ISO3 = {
    'AL': 'ALB', 'AT': 'AUT', 'BE': 'BEL', 'BG': 'BGR',
    'CH': 'CHE', 'CY': 'CYP', 'CZ': 'CZE', 'DE': 'DEU',
    'DK': 'DNK', 'EE': 'EST', 'EL': 'GRC', 'ES': 'ESP',
    'FI': 'FIN', 'FR': 'FRA', 'HR': 'HRV', 'HU': 'HUN',
    'IE': 'IRL', 'IT': 'ITA', 'LT': 'LTU', 'LU': 'LUX',
    'LV': 'LVA', 'ME': 'MNE', 'MT': 'MLT', 'NL': 'NLD',
    'PL': 'POL', 'PT': 'PR1',
    'RO': 'ROU', 'SE': 'SWE',
    'SI': 'SVN', 'SK': 'SVK', 'UK': 'GBR'
}

map_df = (
    final_df[final_df['season'] == 2024]
    .dropna(subset=['quality_score'])
    .groupby('countryCode')
    .apply(lambda x: (x['quality_score'] == 1).sum() / len(x) * 100, include_groups=False)
    .reset_index(name='pct_excellent')
)
map_df['ISO3'] = map_df['countryCode'].map(ALPHA2_TO_ISO3)

countries2 = countries.copy(deep=True)
countries2['pct_excellent'] = map_df.set_index('ISO3')['pct_excellent']

# Draw the map without the default title
fig = px.choropleth(
    countries2,
    geojson=countries2.geometry,
    locations=countries2.index,
    color='pct_excellent',
    range_color=(0, 100),
    color_continuous_scale='RdYlGn',
    labels={'pct_excellent': '% Excellent'},
    hover_name='Name'
)

# 1. FOCUS ON EUROPE
fig.update_geos(
    lonaxis_range=[-20, 45],  # Longitude (cuts off the Atlantic and Asia)
    lataxis_range=[30, 72],   # Latitude (cuts off Africa and the far north)
    visible=True,
    showcoastlines=True,
    coastlinecolor='gray',
    showland=True,
    landcolor='lightgray',
    showframe=False,
    showcountries=True,
    countrycolor='white'
)

# 2. LAYOUT AND LEGEND ADJUSTMENTS
fig.update_layout(
    height=600,
    width=1100,
    margin={"r": 80, "t": 60, "l": 0, "b": 0},

    # Centered title
    title={
        'text': '% Excellent Waters in Europe (2024)',
        'y': 0.95,
        'x': 0.45,
        'xanchor': 'center'
    },

    # Place the legend OUTSIDE the map
    coloraxis_colorbar=dict(
        title="% Excellent",
        x=1.02,
        xanchor='left',
        y=0.5,
        len=0.75,
        thickness=15
    )
)

fig.show()

## 11. Average Water Quality Trend (1990–2024)

Line chart of the average quality score across all of Europe for each year. The y-axis is
inverted so that improving quality (a lower score) reads as an upward trend
(Figure 3 in the report).

In [ ]:
# ============================================================
# CELL 11: Average Water Quality Trend (1990 - 2024)
# ============================================================
yearly_trend = final_df.groupby('season')['quality_score'].mean().reset_index()

sns.set_style("white")

plt.figure(figsize=(12, 6))
sns.lineplot(data=yearly_trend, x='season', y='quality_score', color='#3498db', linewidth=2.5)
plt.gca().invert_yaxis()
plt.title("Average Water Quality Trend in Europe (1990–2024)", fontsize=14)
plt.xlabel("")
plt.ylabel("Average Quality (1 = Excellent)")
plt.tight_layout()
plt.savefig("quality_trend.png", dpi=300, bbox_inches='tight')
plt.show()

## 12. Heatmap of Average Water Quality by Country and Year

Pivot the average quality score into a country x year grid and visualize it as a heatmap,
making it easy to spot long-term trends and outlier years for individual countries
(Figure 4 in the report).

In [ ]:
# ============================================================
# CELL 12: Heatmap of Average Water Quality by Country and Year
# ============================================================

# Compute the average quality score for each country and year
average_quality_by_country_year = final_df.groupby(['season', 'countryCode'])['quality_score'].mean().unstack()

# Map country codes to full names for readability
# Uses COUNTRY_NAMES defined in CELL 6
average_quality_by_country_year.columns = [COUNTRY_NAMES.get(c, c) for c in average_quality_by_country_year.columns]

plt.figure(figsize=(16, 10))
sns.heatmap(
    average_quality_by_country_year.T, # Transpose so years are on the x-axis and countries on the y-axis
    cmap='RdYlGn_r', # Reversed color scale: green is good (lower score), red is bad (higher score)
    linewidths=0.5,
    linecolor='black',
    cbar_kws={'label': 'Average Water Quality Score (1=Excellent, 4=Poor)'},
    annot=False
)

plt.title('Heatmap of Average Water Quality by Country and Year (1990-2024)', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

## 13. Load and Process Coastal Tourism Data

Download the Eurostat dataset on nights spent in coastal accommodation, clean and rename its
columns, and merge it into `final_df` so it can be compared against water quality.

In [ ]:
# ============================================================
# CELL 13: Load and Process Coastal Tourism Data
# ============================================================
TOURISM_FILENAME = "tour_occ_ninatdc__custom_21154463_linear_2_0.csv"

url_tourism = BASE_RAW + TOURISM_FILENAME
df_tour = pd.read_csv(url_tourism)

df_tour_clean = df_tour[['geo', 'TIME_PERIOD', 'OBS_VALUE']].copy()

# Safe conversion to numeric (turns text values like ':' into NaN)
df_tour_clean['OBS_VALUE'] = pd.to_numeric(df_tour_clean['OBS_VALUE'], errors='coerce')

# Rename columns to match the names used in final_df
df_tour_clean.rename(columns={
    'geo': 'countryCode',
    'TIME_PERIOD': 'season',
    'OBS_VALUE': 'coastal_tourism_nights'
}, inplace=True)

# Drop the column first if it already exists (e.g. when re-running this cell)
if 'coastal_tourism_nights' in final_df.columns:
    final_df = final_df.drop(columns=['coastal_tourism_nights'])

# Merge into the main final_df dataset
final_df = pd.merge(
    final_df,
    df_tour_clean,
    on=['countryCode', 'season'],
    how='left'
)

print(f"Current number of columns: {len(final_df.columns)}\n")

## 14. Coastal Tourism vs. Water Quality Over Time

Compare the yearly total of coastal tourism nights across Europe with the average coastal water
quality score, using a dual-axis line chart (Figure 12 in the report).

In [ ]:
# ============================================================
# CELL 14: Coastal Tourism vs. Water Quality Over Time
# ============================================================

# Tourism trend (sum across Europe) and coastal water quality over time
tour_trend = (
    final_df[['countryCode', 'season', 'coastal_tourism_nights']]
    .drop_duplicates()
    .groupby('season')['coastal_tourism_nights']
    .sum()
    .reset_index()
)

qual_trend = (
    final_df[final_df['bathingWaterType'].str.contains('coast', case=False, na=False)]
    .groupby('season')['quality_score']
    .mean()
    .reset_index()
)

# Merge and keep only years that have tourism data
trend_df = (
    pd.merge(tour_trend, qual_trend, on='season')
    .query('coastal_tourism_nights > 0')
    .assign(tourism_mil=lambda x: x['coastal_tourism_nights'] / 1_000_000)
)

# Dual-axis chart: tourism (left axis) + water quality (right axis, inverted)
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(trend_df['season'], trend_df['tourism_mil'], color='#3498db', label='Tourism')
ax1.set_ylabel('Coastal Tourism (million nights)', color='#3498db')
ax1.tick_params(axis='y', labelcolor='#3498db')
ax1.set_xticks(trend_df['season'])
ax1.grid(True, linestyle='--', alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(trend_df['season'], trend_df['quality_score'], color='#2ecc71', linestyle='--', label='Water Quality')
ax2.set_ylabel('Average Water Quality (1=Excellent)', color='#2ecc71')
ax2.tick_params(axis='y', labelcolor='#2ecc71')
ax2.invert_yaxis()

# Shared legend
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left')

plt.title('Coastal Tourism vs. Coastal Water Quality in Europe')
plt.tight_layout()
plt.savefig("tourism_vs_quality.png", dpi=300, bbox_inches='tight')
plt.show()

## 15. Country Deep-Dive: Quality by Type + Map Comparison

Define a reusable function that, for any country, builds a combined figure with three
stacked-area charts (by water type) on top and two side-by-side maps (for a chosen start and end
year) below. Finally, run it for Slovakia, the domestic case discussed in the report.

In [ ]:
# ============================================================
# CELL 15: Country Deep-Dive - Quality by Type + Map Comparison
# ============================================================

def analyze_country(country_code, year1=None, year2=None):
    cc   = country_code.upper()
    df_c = final_df[final_df['countryCode'] == cc].copy()
    if df_c.empty:
        print(f"Country '{cc}' not found.")
        return

    # If no years are given, automatically pick the first and last available year
    name  = COUNTRY_NAMES.get(cc, cc)
    years = sorted(df_c['season'].unique())
    year1 = year1 or years[0]
    year2 = year2 or years[-1]

    ORDER  = ['Excellent', 'Good', 'Good or Sufficient', 'Sufficient', 'Poor']
    COLORS = ['#2ecc71', '#3498db', '#1a6fa8', '#f39c12', '#e74c3c']
    TYPES  = ['Coastal', 'Inland', 'Transitional']

    # Lookup table from Eurostat codes to ISO3 - Choropleth requires ISO3 format
    ALPHA2_TO_ISO3 = {
        'AL':'ALB','AT':'AUT','BE':'BEL','BG':'BGR','CH':'CHE','CY':'CYP',
        'CZ':'CZE','DE':'DEU','DK':'DNK','EE':'EST','EL':'GRC','ES':'ESP',
        'FI':'FIN','FR':'FRA','HR':'HRV','HU':'HUN','IE':'IRL','IT':'ITA',
        'LT':'LTU','LU':'LUX','LV':'LVA','ME':'MNE','MT':'MLT','NL':'NLD',
        'PL':'POL','PT':'PRT','RO':'ROU','SE':'SWE','SI':'SVN','SK':'SVK','UK':'GBR'
    }

    # Translate internal water-type names into readable categories
    df_c['waterType'] = df_c['bathingWaterType'].map(TYPE_MAP)

    # For maps we only need rows with known quality and coordinates
    df_all = df_c[df_c['quality_label'].notna() & df_c['lat'].notna()]

    # -------------------------------------------------------
    # PERCENTAGE CALCULATION FOR STACKED AREA CHARTS
    # groupby + size()        -> count of measurements per combination
    # transform(x / x.sum()) -> percentage share for that year/type
    # -------------------------------------------------------
    pct = (
        df_c.dropna(subset=['quality_label'])
        .groupby(['season', 'waterType', 'quality_label']).size()
        .groupby(level=[0, 1]).transform(lambda x: x / x.sum() * 100)
        .reset_index(name='pct')
    )

    # -------------------------------------------------------
    # LAYOUT: 6-column grid
    # Row 1: 3 charts, each colspan=2
    # Row 2: 2 maps, each colspan=3
    # row_heights -> maps take up 70% of the height
    # -------------------------------------------------------
    fig = make_subplots(
        rows=2, cols=6,
        specs=[
            [{"type":"xy","colspan":2}, None, {"type":"xy","colspan":2}, None, {"type":"xy","colspan":2}, None],
            [{"type":"geo","colspan":3}, None, None, {"type":"geo","colspan":3}, None, None],
        ],
        subplot_titles=TYPES + [str(year1), str(year2)],
        row_heights=[0.3, 0.7],
        vertical_spacing=0.08,
        horizontal_spacing=0.02
    )

    # -------------------------------------------------------
    # ROW 1: STACKED AREA CHARTS
    # fill='tonexty' + stackgroup -> layers stack on top of each other
    # reindex -> fills 0 for years where a category is missing
    # showlegend only for the first chart so the legend doesn't repeat 3 times
    # -------------------------------------------------------
    for idx, (col, wtype) in enumerate(zip([1, 3, 5], TYPES)):
        subset = pct[pct['waterType'] == wtype]
        for quality, color in zip(ORDER, COLORS):
            y = (subset[subset['quality_label'] == quality]
                 .set_index('season')['pct']
                 .reindex(years, fill_value=0))
            fig.add_trace(go.Scatter(
                x=y.index, y=y.values, name=quality,
                fill='tonexty', fillcolor=color, line=dict(color=color, width=0),
                stackgroup='one', legendgroup=quality, showlegend=(idx == 0)
            ), row=1, col=col)

    # -------------------------------------------------------
    # ROW 2: MAPS
    # bounds -> country's coordinate range + a 2 degree buffer around it
    # Choropleth -> shades the country gray using its ISO3 code
    # If a year has no data, df_q will be empty and Plotly draws nothing
    # -------------------------------------------------------
    bounds = dict(
        lonaxis_range=[df_all['lon'].min()-2, df_all['lon'].max()+2],
        lataxis_range=[df_all['lat'].min()-2, df_all['lat'].max()+2],
    )
    geo_settings = dict(showland=True, landcolor='white', showcountries=True,
                        showcoastlines=True, coastlinecolor='gray', showframe=False)

    for map_col, yr in [(1, year1), (4, year2)]:
        fig.add_trace(go.Choropleth(
            locations=[ALPHA2_TO_ISO3.get(cc)], z=[1],
            colorscale=[[0,'#d0d0d0'],[1,'#d0d0d0']],
            showscale=False, showlegend=False, hoverinfo='skip'
        ), row=2, col=map_col)

        df_m = df_all[df_all['season'] == yr]
        for quality, color in zip(ORDER, COLORS):
            df_q = df_m[df_m['quality_label'] == quality]
            fig.add_trace(go.Scattergeo(
                lat=df_q['lat'], lon=df_q['lon'], mode='markers',
                marker=dict(size=8, color=color), name=quality,
                legendgroup=quality, showlegend=False,
                hovertemplate=f"<b>{quality}</b><br>%{{text}}<extra></extra>",
                text=df_q['bathingWaterType']
            ), row=2, col=map_col)

    # 'geo' = first map, 'geo2' = second - Plotly numbers them automatically
    for geo_id in ['geo', 'geo2']:
        fig.update_layout(**{geo_id: {**geo_settings, **bounds}})

    fig.update_yaxes(range=[0, 100], row=1)
    fig.update_layout(
        title=dict(text=f"Water Quality — {name} ({year1} vs {year2})",
                   font=dict(size=16), x=0.5, xanchor='center'),
        height=1050, width=1400,
        legend=dict(x=1.01, y=0.95, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0.8)',
                    bordercolor='rgba(0,0,0,0.15)', borderwidth=1, font=dict(size=12))
    )
    fig.show()


analyze_country('SK', 2000, 2024)